In [ ]:
# RESUME_CLASSIFICATION_PROJECT_CHROMADB/
# │
# ├── Education/          # 5000 resume PDFs
# ├── JD_Matches/         # job descriptions
# ├── chroma_db/          # ChromaDB will be saved here later
# ├── models/             # ML models will be saved here
# └── Resume_Classification_and_Matching_System.ipynb

In [25]:
import os
import fitz
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

import joblib

In [26]:
print(os.getcwd())
print(os.listdir())

print("Total resumes:", len(os.listdir("Education")))

c:\Users\AFISHA AZME\OneDrive\Documents\RESUME_CLASSIFICATION_PROJECT_CHROMADB
['.venv', 'chroma_db', 'Education', 'JD_Matches', 'models', 'Resume_Classification_and_Matching_System.ipynb']
Total resumes: 5000


In [27]:
def extract_text(pdf_path):
    text = ""

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text()

    except Exception as e:
        print("Error:", pdf_path)

    return text.strip()

In [28]:
resume_names = []
resume_texts = []

for file in os.listdir("Education"):

    if file.lower().endswith(".pdf"):

        path = os.path.join("Education", file)

        text = extract_text(path)

        if len(text) > 100:
            resume_names.append(file)
            resume_texts.append(text)

print("Loaded resumes:", len(resume_names))
print("Sample resume text:", resume_texts[0][:500])

Loaded resumes: 5000
Sample resume text: Lori Hart
lori.hart30001@reed-kennedy.com | (538)274-8914x081 | 4458 Perez Mountains Suite 094, Lake Debra, MH
88430 | linkedin.com/in/lori-hart-30001
PROFESSIONAL SUMMARY
Experienced professional with a strong background in Education and expertise in Agile, Python, AWS. Proven
track record of delivering high-quality results in fast-paced environments. Excellent Leadership, Attention to
Detail skills with a passion for continuous learning and professional development.
PROFESSIONAL EXPERIENCE
Sof


In [29]:
df = pd.DataFrame({
    "resume_name": resume_names,
    "text": resume_texts
})

df.head()

,resume_name,text
0,Resume_030001_Lori_Hart.pdf,Lori Hart\nlori.hart30001@reed-kennedy.com | (...
1,Resume_030002_Anthony_Moses.pdf,Anthony Moses\nanthony.moses30002@scott.com | ...
2,Resume_030003_Edward_Shepherd.pdf,Edward Shepherd\nedward.shepherd30003@marshall...
3,Resume_030004_Nicholas_Potter.pdf,Nicholas Potter\nnicholas.potter30004@wiley.or...
4,Resume_030005_Donna_Watts.pdf,Donna Watts\ndonna.watts30005@wells-gonzalez.c...


In [30]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X = tfidf.fit_transform(df["text"])

print("TF-IDF shape:", X.shape)

TF-IDF shape: (5000, 5000)


In [31]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

df.head()

,resume_name,text,cluster
0,Resume_030001_Lori_Hart.pdf,Lori Hart\nlori.hart30001@reed-kennedy.com | (...,2
1,Resume_030002_Anthony_Moses.pdf,Anthony Moses\nanthony.moses30002@scott.com | ...,1
2,Resume_030003_Edward_Shepherd.pdf,Edward Shepherd\nedward.shepherd30003@marshall...,1
3,Resume_030004_Nicholas_Potter.pdf,Nicholas Potter\nnicholas.potter30004@wiley.or...,3
4,Resume_030005_Donna_Watts.pdf,Donna Watts\ndonna.watts30005@wells-gonzalez.c...,2


In [32]:
df["cluster"].value_counts()

cluster
2    1814
3    1159
1    1120
0     779
4     128
Name: count, dtype: int64

In [33]:
terms = tfidf.get_feature_names_out()

for i in range(5):
    center = kmeans.cluster_centers_[i]
    top_words = np.array(terms)[center.argsort()[-20:]]

    print("\nCluster", i)
    print(top_words)


Cluster 0
['node' 'scrum' 'gpa' 'university' 'excellent' 'analysis' 'time' 'project'
 'strong' 'learning' 'management' 'architect' 'solutions' 'certified'
 'data' 'technologies' 'education' 'aws' 'skills' 'professional']

Cluster 1
['communication' 'aws' 'machine' 'ci' 'cd' 'gpa' 'university' 'scrum'
 'excellent' 'time' 'data' 'project' 'strong' 'azure' 'learning'
 'management' 'technologies' 'education' 'skills' 'professional']

Cluster 2
['analysis' 'manager' 'machine' 'engineer' 'gpa' 'university' 'scrum'
 'excellent' 'azure' 'devops' 'time' 'learning' 'strong' 'project'
 'management' 'data' 'technologies' 'education' 'skills' 'professional']

Cluster 3
['azure' 'python' 'machine' 'kubernetes' 'project' 'scrum' 'devops'
 'excellent' 'time' 'analysis' 'university' 'gpa' 'strong' 'learning'
 'management' 'data' 'technologies' 'education' 'skills' 'professional']

Cluster 4
['azure' 'javascript' 'devops' 'git' 'scrum' 'time' 'docker' 'gpa'
 'university' 'excellent' 'strong' 'project' 

In [34]:
print(df["cluster"].value_counts())

cluster
2    1814
3    1159
1    1120
0     779
4     128
Name: count, dtype: int64


terms = tfidf.get_feature_names_out()

for i in range(5):
    center = kmeans.cluster_centers_[i]
    top_words = np.array(terms)[center.argsort()[-20:]]

    print("\nCluster", i)
    print(top_words)

In [41]:
custom_stop_words = [
    "professional", "skills", "education", "technologies",
    "excellent", "strong", "university", "gpa",
    "project", "time", "learning", "management",
    "experience", "background", "results"
]

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    max_df=0.70,
    min_df=5
)

X = tfidf.fit_transform(df["text"])

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print(df["cluster"].value_counts())

cluster
4    2008
1    1864
0     854
3     169
2     105
Name: count, dtype: int64


In [42]:
terms = tfidf.get_feature_names_out()

for i in range(5):
    center = kmeans.cluster_centers_[i]
    top_words = np.array(terms)[center.argsort()[-25:]]

    print("\nCluster", i)
    print(top_words)


Cluster 0
['product' 'ph' 'administrator' '2019' 'master' 'christopher' 'analyst'
 'ethic' 'adaptability' 'leadership' 'certified' 'science' 'teamwork'
 'business' 'manager' 'critical' 'thinking' 'work' 'creativity' 'engineer'
 'attention' '2023' '2022' 'certification' 'pmp']

Cluster 1
['conversational' 'manager' 'native' 'ph' 'fluent' 'bachelor' 'basic'
 'associate' 'engineer' 'administrator' 'business' 'science' 'teamwork'
 'adaptability' 'creativity' 'leadership' 'critical' 'ethic' 'thinking'
 '2022' 'attention' 'master' 'work' '2023' 'certified']

Cluster 2
['scientist' 'cloud' 'google' 'administrator' 'group' 'adaptability'
 'source' 'master' 'ethic' 'creativity' 'business' 'product' 'leadership'
 'basic' '2023' 'critical' 'teamwork' 'attention' 'manager' 'thinking'
 'work' 'engineer' '2022' 'certified' 'james']

Cluster 3
['scientist' 'analyst' 'native' 'fluent' 'master' 'conversational'
 'business' 'administrator' 'creativity' 'leadership' 'ethic' 'critical'
 'thinking' 'teamw

In [43]:
df["category"] = "Cluster_" + df["cluster"].astype(str)

df.head()

,resume_name,text,cluster,category
0,Resume_030001_Lori_Hart.pdf,Lori Hart\nlori.hart30001@reed-kennedy.com | (...,4,Cluster_4
1,Resume_030002_Anthony_Moses.pdf,Anthony Moses\nanthony.moses30002@scott.com | ...,0,Cluster_0
2,Resume_030003_Edward_Shepherd.pdf,Edward Shepherd\nedward.shepherd30003@marshall...,1,Cluster_1
3,Resume_030004_Nicholas_Potter.pdf,Nicholas Potter\nnicholas.potter30004@wiley.or...,1,Cluster_1
4,Resume_030005_Donna_Watts.pdf,Donna Watts\ndonna.watts30005@wells-gonzalez.c...,4,Cluster_4


In [44]:
df["category"].value_counts()

category
Cluster_4    2008
Cluster_1    1864
Cluster_0     854
Cluster_3     169
Cluster_2     105
Name: count, dtype: int64

In [45]:
import joblib

joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
joblib.dump(kmeans, "models/kmeans_model.pkl")

df.to_csv("clustered_resumes.csv", index=False)

print("Saved model and clustered resumes")

Saved model and clustered resumes


Task 2: Create ChromaDB collection

In [46]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection(
    name="resume_collection"
)

print("Collection created")

Collection created


In [47]:
ids = []
documents = []
metadatas = []

for i, row in df.iterrows():

    ids.append(str(i))

    documents.append(row["text"][:5000])

    metadatas.append({
        "resume_name": row["resume_name"],
        "category": row["category"],
        "cluster": int(row["cluster"])
    })

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Total documents in ChromaDB:", collection.count())

C:\Users\AFISHA AZME\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.51MiB/s]


Total documents in ChromaDB: 5000


In [48]:
results = collection.get(
    where={"category": "Cluster_0"},
    limit=3
)

results["metadatas"]

[{'resume_name': 'Resume_030002_Anthony_Moses.pdf',
  'category': 'Cluster_0',
  'cluster': 0},
 {'category': 'Cluster_0',
  'cluster': 0,
  'resume_name': 'Resume_030008_Alexander_Huang.pdf'},
 {'resume_name': 'Resume_030011_Kyle_George.pdf',
  'category': 'Cluster_0',
  'cluster': 0}]

In [49]:
def match_jd_to_resumes(job_description, top_k=3):

    jd_vector = tfidf.transform([job_description])

    predicted_cluster = kmeans.predict(jd_vector)[0]

    predicted_category = "Cluster_" + str(predicted_cluster)

    results = collection.query(
        query_texts=[job_description],
        n_results=top_k,
        where={"category": predicted_category}
    )

    print("Predicted Category:", predicted_category)
    print("\nTop 3 Matched Resumes:")

    for i in range(top_k):
        print("\nRank:", i + 1)
        print("Resume:", results["metadatas"][0][i]["resume_name"])
        print("Category:", results["metadatas"][0][i]["category"])
        print("Distance:", results["distances"][0][i])

    return results

In [50]:
jd = """
We are hiring a DevOps Engineer with Docker, Kubernetes, Jenkins,
CI/CD, AWS, Azure, Linux and cloud deployment experience.
"""

match_jd_to_resumes(jd)

Predicted Category: Cluster_1

Top 3 Matched Resumes:

Rank: 1
Resume: Resume_032481_Christopher_Smith.pdf
Category: Cluster_1
Distance: 0.8307478427886963

Rank: 2
Resume: Resume_032575_Kevin_Armstrong.pdf
Category: Cluster_1
Distance: 0.8319547176361084

Rank: 3
Resume: Resume_031306_Devin_Wallace.pdf
Category: Cluster_1
Distance: 0.8547893166542053


{'ids': [['2480', '2574', '1305']],
 'embeddings': None,
 'documents': [["Christopher Smith\nchristopher.smith32481@haynes-navarro.com | +1-431-952-0373x856 | 184 Brandy Isle, Hillshire, SD 36188 |\nlinkedin.com/in/christopher-smith-32481\nPROFESSIONAL SUMMARY\nExperienced professional with a strong background in Education and expertise in Docker, Scrum, Git. Proven\ntrack record of delivering high-quality results in fast-paced environments. Excellent Critical Thinking,\nCommunication skills with a passion for continuous learning and professional development.\nPROFESSIONAL EXPERIENCE\nDevOps Engineer | Thompson and Sons | 2022 - 2024\n• Finally everyone read kind word research defense.\n• Father look or mouth.\n• House scene sometimes until their outside style past.\n• Behavior production wrong future let.\nDevOps Engineer | Lee, Ray and Hudson | 2018 - Present\n• Though population cold receive four happen.\n• Authority woman economic story seek.\n• However compare order team strong ru